# 01 — Validate Bronze raw and structured landing

AIDP operational pipeline notebook. It uses Volumes and governed tables only; the Git checkout is code only.

In [ ]:
from pathlib import Path
import os
import sys
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
root = Path.cwd().resolve()
for candidate in (Path(os.environ['AIDP_PROJECT_ROOT']).resolve(), root, *root.parents) if os.getenv('AIDP_PROJECT_ROOT') else (root, *root.parents):
    if (candidate / 'src').is_dir() and (candidate / 'config/settings.yaml').is_file():
        root = candidate
        break
else:
    raise RuntimeError('Project code root not found. Clone the repository; do not use it as a data store.')
sys.path.insert(0, str(root / 'src'))
from aidp_pipeline import LANDING_COUNTS, RAW_SESSIONS, STRUCTURAL_KEYS, landing_paths, load_aidp_config, missing_landing_paths, table_name
config = load_aidp_config(root / 'config/settings.yaml')


In [ ]:
from pathlib import Path

for path in [
    "/Volumes/race_intelligence/bronze",
    "/Volumes/race_intelligence/bronze/raw_timing",
    "/Volumes/race_intelligence/bronze/validated_landing",
]:
    p = Path(path)
    print("\n", path)
    print("exists:", p.exists())
    if p.exists():
        print(list(p.iterdir())[:20])

In [ ]:
raw = Path(config['bronze_raw_volume'])
raw_rows = []
for session in RAW_SESSIONS:
    files = [p for p in (raw / session).rglob('*') if p.is_file()] if (raw / session).is_dir() else []
    raw_rows.append((session, len(files), sum(p.suffix.lower() == '.csv' for p in files), sum(p.suffix.lower() == '.pdf' for p in files)))
spark.createDataFrame(raw_rows, 'session string, file_count long, csv_count long, pdf_count long').show(truncate=False)
missing = missing_landing_paths(config, lambda path: Path(path).is_file())
if missing:
    from aidp_pipeline import landing_prerequisite_message
    raise RuntimeError(landing_prerequisite_message(config, missing))


In [ ]:
from aidp_pipeline import validate_landing_with_spark
summary = spark.createDataFrame(validate_landing_with_spark(spark, config))
summary.orderBy('dataset').show(truncate=False)
if summary.filter(~F.col('row_count_passed') | (F.col('missing_key_columns') != '') | (F.col('structural_key_null_rows') > 0)).count():
    raise ValueError('Bronze structured landing validation failed; do not promote to Silver.')
print('Validated structured Bronze landing is ready for promotion.')
